# AqSolDB — Hybrid Structure-Aware Split

`AqSolDB_representations.ipynb` used a pure Bemis–Murcko scaffold split with
largest-scaffold-first assignment. That split was scaffold-disjoint, but strongly
imbalanced: train held 7,984 molecules across only 297 scaffolds while test held 1,996
across 1,650 (1,343 of them singletons), and one CV fold contained 2,940 molecules from
a single scaffold.

The cause is the empty Murcko scaffold. It is not a scaffold — it is every acyclic
molecule in the database (2,940, or 29.5%) grouped together because none has a ring
system. Murcko disjointness exists to stop a shared *core* spanning the split; acyclic
molecules share no core, only the absence of one. Since the block is indivisible, no
fold-assignment algorithm can fix that 36.8% fold.

This notebook groups cyclic molecules by Murcko scaffold and acyclic molecules by
Butina fingerprint-similarity cluster, then assigns whole groups to train/test with a
balance objective. It is a **hybrid structure-aware split**, not a pure Bemis–Murcko
split, and is named that way throughout.

The old results are kept as a separate baseline. The goal is evaluation balance, not a
lower RMSE; the split is chosen on structural criteria only, before any model is fitted.

## 1. Imports and constants

Every tunable lives here so it is easy to find and change.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.ML.Cluster import Butina

from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from xgboost import XGBRegressor

RDLogger.DisableLog("rdApp.*")      # AqSolDB triggers many harmless valence warnings

RANDOM_STATE = 42
TEST_FRACTION = 0.20
N_FOLDS = 5

# --- acyclic clustering ---
# Butina works on DISTANCE, so cutoff = 1 - similarity.
ACYCLIC_SIMILARITY_THRESHOLD = 0.5     # Tanimoto similarity; see Section 4 for how this was chosen
MORGAN_RADIUS = 2
MORGAN_NBITS = 2048

# --- balanced splitter ---
N_SPLIT_TRIALS = 300                   # random restarts
CONCENTRATION_PENALTY_WEIGHT = 1.0     # weight on the "one group dominates a side" penalty

# Publication-quality defaults: readable fonts, vector output.
plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "legend.frameon": False,
})

FIGURE_PREFIX = "bsplit_"   # figures are written next to this notebook, no subfolder


def save_figure(fig, name):
    # Save as vector PDF beside the notebook.
    path = f"{FIGURE_PREFIX}{name}.pdf"
    fig.savefig(path, bbox_inches="tight")
    print(f"saved {path}")


print("Setup complete.")

## 2. Load AqSolDB

Tab-separated despite the `.csv` extension. We keep only SMILES and the target and
recompute everything ourselves, so the pipeline stays identical to the ESOL work.

In [ ]:
df = pd.read_csv("aqsoldb.csv", sep="\t")[["SMILES", "Solubility"]]
print("Rows initially :", len(df))

# Parse every SMILES once and keep the Mol objects — we reuse them for scaffolds,
# fingerprints and descriptors, so parsing once saves a lot of time.
mols = [Chem.MolFromSmiles(s) for s in df["SMILES"]]
keep = [i for i, m in enumerate(mols) if m is not None]
df = df.iloc[keep].reset_index(drop=True)
mols = [mols[i] for i in keep]

target_ok = df["Solubility"].notna().values
df = df[target_ok].reset_index(drop=True)
mols = [m for m, k in zip(mols, target_ok) if k]

y_all = df["Solubility"].values
print("Rows after cleaning :", len(df))
df.head(3)

## 3. Murcko scaffold distribution

Before changing anything, we measure exactly what the old grouping looks like.

In [ ]:
def get_murcko_scaffold(mol):
    """Bemis-Murcko scaffold SMILES. Returns '' for acyclic molecules."""
    try:
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    except Exception:
        return ""

df["scaffold"] = [get_murcko_scaffold(m) for m in mols]

scaffold_counts = df["scaffold"].value_counts()
n_total = len(df)

print(f"Total molecules        : {n_total}")
print(f"Unique Murcko scaffolds: {len(scaffold_counts)}")
print(f"Singleton scaffolds    : {int((scaffold_counts == 1).sum())}")
print(f"Empty (acyclic) group  : {int(scaffold_counts.get('', 0))} molecules "
      f"= {100 * scaffold_counts.get('', 0) / n_total:.1f}% of the dataset")

print("\nLargest 20 scaffolds:")
for scaf, count in scaffold_counts.head(20).items():
    name = scaf if scaf else "<EMPTY = acyclic>"
    print(f"  {count:6d}  {100 * count / n_total:5.2f}%   {name[:55]}")

In [ ]:
# Fraction of molecules covered by the largest N scaffolds
cumulative = np.cumsum(scaffold_counts.values) / n_total
print("Fraction of all molecules contained in the largest N scaffolds:")
for k in (1, 2, 5, 10, 20):
    print(f"  top {k:3d} scaffolds -> {100 * cumulative[k - 1]:.1f}%")

print("\nScaffold size buckets:")
for low, high in [(1, 1), (2, 2), (3, 5), (6, 10), (11, 50), (51, 200), (201, 10**9)]:
    selected = scaffold_counts[(scaffold_counts >= low) & (scaffold_counts <= high)]
    label = f"{low}-{high}" if high < 10**9 else f"{low}+"
    print(f"  size {label:>7}: {len(selected):5d} scaffolds, "
          f"{int(selected.sum()):5d} molecules ({100 * selected.sum() / n_total:5.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(scaffold_counts.values, bins=60, log=True)
axes[0].set_xlabel("Molecules per scaffold")
axes[0].set_ylabel("Number of scaffolds (log)")
axes[0].set_title("Murcko scaffold size distribution")

axes[1].plot(np.arange(1, len(cumulative) + 1), cumulative)
axes[1].axhline(0.8, color="r", linestyle="--", linewidth=1, label="80% coverage")
axes[1].set_xlabel("Number of scaffolds (largest first)")
axes[1].set_ylabel("Cumulative fraction of molecules")
axes[1].set_title("Scaffold cumulative coverage")
axes[1].legend()

plt.tight_layout()
save_figure(fig, "01_scaffold_distribution")
plt.show()

## 4. Cluster the acyclic molecules by fingerprint similarity

For acyclic molecules only, we compute Morgan fingerprints (radius 2, 2048 bits),
build the pairwise Tanimoto distance matrix, and cluster with RDKit's Butina
algorithm. Cyclic molecules keep their ordinary Murcko scaffold labels — untouched.

In [ ]:
acyclic_positions = np.where(df["scaffold"].values == "")[0]
print(f"Acyclic molecules to cluster: {len(acyclic_positions)}")

morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=MORGAN_RADIUS, fpSize=MORGAN_NBITS
)
acyclic_fps = [morgan_generator.GetFingerprint(mols[i]) for i in acyclic_positions]

# Butina wants the lower triangle of the DISTANCE matrix as one flat list.
# Tanimoto distance = 1 - Tanimoto similarity.
t0 = time.time()
distance_matrix = []
for i in range(1, len(acyclic_fps)):
    sims = DataStructs.BulkTanimotoSimilarity(acyclic_fps[i], acyclic_fps[:i])
    distance_matrix.extend(1.0 - s for s in sims)
print(f"Pairwise distances: {len(distance_matrix):,} pairs in {time.time() - t0:.1f}s")

### Choosing the similarity threshold

The threshold must not be picked from model error. The rule used here is structural:
choose the threshold giving acyclic clusters roughly the granularity Murcko scaffolds
already give cyclic molecules. Much coarser over-merges unrelated molecules; much finer
makes nearly every acyclic its own group, degenerating toward a random split for 30% of
the data.

Measure the cyclic reference first, then sweep and compare.

In [ ]:
cyclic_sizes = df[df["scaffold"] != ""]["scaffold"].value_counts().values

print("REFERENCE — Murcko grouping on cyclic molecules:")
print(f"  molecules          : {cyclic_sizes.sum()}")
print(f"  groups             : {len(cyclic_sizes)}")
print(f"  singleton fraction : {100 * (cyclic_sizes == 1).mean():.1f}%")
print(f"  median size        : {int(np.median(cyclic_sizes))}")
print(f"  90th percentile    : {int(np.percentile(cyclic_sizes, 90))}")
# The mean is dominated by the single benzene group, so we also report it without benzene.
print(f"  mean size          : {cyclic_sizes.mean():.2f} "
      f"(excluding benzene: {cyclic_sizes[1:].mean():.2f})")

In [ ]:
sweep_rows = []
for candidate_sim in (0.8, 0.7, 0.65, 0.6, 0.55, 0.5, 0.4):
    trial_clusters = Butina.ClusterData(
        distance_matrix, len(acyclic_fps), 1.0 - candidate_sim, isDistData=True
    )
    sizes = np.array(sorted((len(c) for c in trial_clusters), reverse=True))
    sweep_rows.append({
        "similarity": candidate_sim,
        "clusters": len(trial_clusters),
        "singleton %": round(100 * (sizes == 1).mean(), 1),
        "median": int(np.median(sizes)),
        "mean": round(float(sizes.mean()), 2),
        "p90": int(np.percentile(sizes, 90)),
        "largest": int(sizes[0]),
    })

threshold_sweep = pd.DataFrame(sweep_rows)
print("CANDIDATE — Butina clusters on acyclic molecules, swept over thresholds:\n")
print(threshold_sweep.to_string(index=False))

Against the cyclic reference (69.0% singletons, median 1, p90 4, mean 2.72 excluding
benzene), **0.5** matches on every statistic: 63.3% singletons, median 1, p90 4, mean
2.46. The conventional ECFP4 analogue threshold of 0.7 gives 85% singletons — far finer
than Murcko grouping. At 0.4 over-merging begins (p90 jumps to 7, largest cluster 237).
Erring lower is also the conservative direction, since larger clusters keep more similar
molecules on the same side.

Every threshold in the sweep dissolves the 2,940 block, so this is a choice about
granularity, not about whether the fix works.

In [ ]:
acyclic_clusters = Butina.ClusterData(
    distance_matrix, len(acyclic_fps), 1.0 - ACYCLIC_SIMILARITY_THRESHOLD, isDistData=True
)

cluster_sizes = np.array(sorted((len(c) for c in acyclic_clusters), reverse=True))
print(f"Threshold: Tanimoto similarity >= {ACYCLIC_SIMILARITY_THRESHOLD} "
      f"(Butina distance cutoff {1 - ACYCLIC_SIMILARITY_THRESHOLD})\n")
print(f"Number of acyclic clusters : {len(acyclic_clusters)}")
print(f"Median cluster size        : {int(np.median(cluster_sizes))}")
print(f"Largest cluster size       : {cluster_sizes[0]} "
      f"({100 * cluster_sizes[0] / n_total:.1f}% of the dataset)")
print(f"Singleton clusters         : {int((cluster_sizes == 1).sum())}")
print(f"Top-10 cluster sizes       : {list(cluster_sizes[:10])}")

In [ ]:
largest_cluster = max(acyclic_clusters, key=len)
print(f"Largest acyclic cluster has {len(largest_cluster)} molecules. First 10 SMILES:\n")
for local_index in largest_cluster[:10]:
    print("  ", df["SMILES"].iloc[acyclic_positions[local_index]])

## 5. Build the hybrid group labels

Cyclic molecules keep their Murcko scaffold string. Acyclic molecules get
`acyclic_cluster_0`, `acyclic_cluster_1`, ... These labels then behave exactly like
ordinary scaffold groups everywhere downstream.

In [ ]:
group_labels = df["scaffold"].tolist()          # start from the Murcko labels
for cluster_id, members in enumerate(acyclic_clusters):
    for local_index in members:
        group_labels[acyclic_positions[local_index]] = f"acyclic_cluster_{cluster_id}"

df["group"] = group_labels

# Map each group label -> array of row positions. This is the object the splitter uses.
scaffold_to_indices = {}
for row_position, group in enumerate(df["group"]):
    scaffold_to_indices.setdefault(group, []).append(row_position)
scaffold_to_indices = {g: np.array(v) for g, v in scaffold_to_indices.items()}

n_cyclic_groups = df.loc[df["scaffold"] != "", "scaffold"].nunique()
print(f"Hybrid groups total : {len(scaffold_to_indices)}")
print(f"  cyclic  (Murcko)  : {n_cyclic_groups}")
print(f"  acyclic (Butina)  : {len(acyclic_clusters)}")

hybrid_sizes = np.array(sorted((len(v) for v in scaffold_to_indices.values()), reverse=True))
print(f"\nLargest hybrid groups : {list(hybrid_sizes[:8])}")
print(f"Singleton groups      : {int((hybrid_sizes == 1).sum())}")
print(f"\nThe old 2,940-molecule block is gone. The largest remaining group is benzene "
      f"({hybrid_sizes[0]}),\nwhich is a genuine shared scaffold and stays intact by design.")

## 6. Reconstruct the OLD split (baseline for comparison)

We rebuild the previous pure-Murcko largest-first split so the two can be compared
side by side. This does not modify `AqSolDB_representations.ipynb`.

In [ ]:
def largest_first_split(groups_dict, n_total, test_fraction):
    """The OLD splitter: fill train with the biggest groups first, rest goes to test."""
    groups_sorted = sorted(groups_dict.values(), key=len, reverse=True)
    n_train_target = int(round(n_total * (1 - test_fraction)))
    train_idx, test_idx = [], []
    for group in groups_sorted:
        if len(train_idx) + len(group) <= n_train_target:
            train_idx.extend(group)
        else:
            test_idx.extend(group)
    return np.array(sorted(train_idx)), np.array(sorted(test_idx))

murcko_to_indices = {}
for row_position, scaf in enumerate(df["scaffold"]):
    murcko_to_indices.setdefault(scaf, []).append(row_position)

train_idx_old, test_idx_old = largest_first_split(murcko_to_indices, n_total, TEST_FRACTION)
print(f"OLD split -> train {len(train_idx_old)}, test {len(test_idx_old)}")

## 7. Balanced group-aware splitter

Shuffle the hybrid groups with a seeded RNG, give each whole group to whichever side is
furthest below its molecule quota, score the candidate, and keep the best over
`N_SPLIT_TRIALS` restarts.

```text
score = |train molecule fraction - 0.80|
      + |train group fraction    - 0.80|
      + CONCENTRATION_PENALTY_WEIGHT * (largest group's share of its own side)
```

Structural criteria only — no target values, no model. The third term prevents benzene
(1,753 molecules) landing in test, where it would be 88% of the test set and absent from
training; that candidate scores badly and is rejected without a special-case rule.

In [ ]:
group_names = list(scaffold_to_indices.keys())
group_sizes = np.array([len(scaffold_to_indices[g]) for g in group_names])
n_groups = len(group_names)

train_molecule_quota = n_total * (1 - TEST_FRACTION)
test_molecule_quota = n_total * TEST_FRACTION


def score_candidate_split(train_groups, test_groups):
    """Lower is better. Structural criteria only — never uses y or model performance."""
    n_train = sum(group_sizes[i] for i in train_groups)
    n_test = sum(group_sizes[i] for i in test_groups)
    if n_train == 0 or n_test == 0:
        return np.inf

    molecule_imbalance = abs(n_train / n_total - (1 - TEST_FRACTION))
    group_imbalance = abs(len(train_groups) / n_groups - (1 - TEST_FRACTION))
    concentration = max(
        max(group_sizes[i] for i in train_groups) / n_train,
        max(group_sizes[i] for i in test_groups) / n_test,
    )
    return molecule_imbalance + group_imbalance + CONCENTRATION_PENALTY_WEIGHT * concentration


split_search_t0 = time.time()

best_score = np.inf
best_train_groups, best_test_groups = None, None
seed_generator = np.random.default_rng(RANDOM_STATE)

for trial in range(N_SPLIT_TRIALS):
    rng = np.random.default_rng(seed_generator.integers(1 << 31))
    shuffled_order = rng.permutation(n_groups)

    train_groups, test_groups = [], []
    n_train, n_test = 0, 0
    for group_index in shuffled_order:
        # "Furthest below quota" measured as a fraction of that side's quota,
        # so the 80/20 target is respected rather than raw molecule counts.
        if (n_train / train_molecule_quota) <= (n_test / test_molecule_quota):
            train_groups.append(group_index)
            n_train += group_sizes[group_index]
        else:
            test_groups.append(group_index)
            n_test += group_sizes[group_index]

    candidate_score = score_candidate_split(train_groups, test_groups)
    if candidate_score < best_score:
        best_score = candidate_score
        best_train_groups, best_test_groups = train_groups, test_groups

train_idx_balanced = np.sort(np.concatenate(
    [scaffold_to_indices[group_names[i]] for i in best_train_groups]))
test_idx_balanced = np.sort(np.concatenate(
    [scaffold_to_indices[group_names[i]] for i in best_test_groups]))

print(f"Split search runtime: {time.time() - split_search_t0:.1f} s")
print(f"Best score over {N_SPLIT_TRIALS} trials: {best_score:.4f}")
print(f"train_idx_balanced: {len(train_idx_balanced)} molecules")
print(f"test_idx_balanced : {len(test_idx_balanced)} molecules")

## 8. Verify disjointness and compare against the old split

The single most important check: no group may appear on both sides.

In [ ]:
groups_train = df["group"].iloc[train_idx_balanced].values
groups_test = df["group"].iloc[test_idx_balanced].values

train_scaffolds = set(groups_train)
test_scaffolds = set(groups_test)

print("Intersection of train and test groups:")
print(train_scaffolds.intersection(test_scaffolds))
print()
print("Is the intersection empty?", train_scaffolds.isdisjoint(test_scaffolds))
assert train_scaffolds.isdisjoint(test_scaffolds), "Group leakage between train and test!"

In [ ]:
def describe_split(name, train_idx, test_idx, group_column):
    """Collect the statistics we want to compare between the old and new splits."""
    train_groups = df[group_column].iloc[train_idx]
    test_groups = df[group_column].iloc[test_idx]
    train_counts = train_groups.value_counts()
    test_counts = test_groups.value_counts()
    return {
        "Split": name,
        "train molecules": len(train_idx),
        "test molecules": len(test_idx),
        "train groups": len(train_counts),
        "test groups": len(test_counts),
        "% groups in train": round(100 * len(train_counts) / (len(train_counts) + len(test_counts)), 1),
        "train singletons": int((train_counts == 1).sum()),
        "test singletons": int((test_counts == 1).sum()),
        "top-10 share train": round(train_counts.head(10).sum() / len(train_idx), 3),
        "top-10 share test": round(test_counts.head(10).sum() / len(test_idx), 3),
        "largest train group": int(train_counts.iloc[0]),
        "largest test group": int(test_counts.iloc[0]),
        "group overlap": len(set(train_counts.index) & set(test_counts.index)),
    }

split_comparison = pd.DataFrame([
    describe_split("OLD  pure Murcko, largest-first", train_idx_old, test_idx_old, "scaffold"),
    describe_split("NEW  hybrid, balanced", train_idx_balanced, test_idx_balanced, "group"),
]).set_index("Split").T

print(split_comparison.to_string())

In [ ]:
print("Top-10 group sizes\n")
old_train_counts = df["scaffold"].iloc[train_idx_old].value_counts()
old_test_counts = df["scaffold"].iloc[test_idx_old].value_counts()
new_train_counts = df["group"].iloc[train_idx_balanced].value_counts()
new_test_counts = df["group"].iloc[test_idx_balanced].value_counts()

print(f"OLD train : {list(old_train_counts.head(10).values)}")
print(f"OLD test  : {list(old_test_counts.head(10).values)}")
print(f"NEW train : {list(new_train_counts.head(10).values)}")
print(f"NEW test  : {list(new_test_counts.head(10).values)}")

## 9. Group-balanced cross-validation folds

`GroupKFold(5)` is valid but produced very uneven folds. Instead: sort training groups
largest first and assign each whole group to whichever fold currently has the fewest
molecules — the standard longest-processing-time heuristic. Groups stay intact, so
disjointness holds by construction, and fold *size* is targeted directly.

`StratifiedGroupKFold` balances label distribution rather than fold size, so it does not
address this problem; it would also require binning a continuous target. The split stays
independent of `y`, and the fold target distribution is reported below as a check.

In [ ]:
def greedy_balanced_folds(train_idx, group_series, n_folds):
    """Assign whole groups to folds, largest group first, always to the smallest fold."""
    group_to_rows = {}
    for row_position in train_idx:
        group_to_rows.setdefault(group_series.iloc[row_position], []).append(row_position)

    groups_by_size = sorted(group_to_rows.items(), key=lambda kv: len(kv[1]), reverse=True)

    folds = [[] for _ in range(n_folds)]
    for group_name, member_rows in groups_by_size:
        smallest_fold = min(range(n_folds), key=lambda k: len(folds[k]))
        folds[smallest_fold].extend(member_rows)

    return [np.array(sorted(f)) for f in folds]


validation_folds = greedy_balanced_folds(train_idx_balanced, df["group"], N_FOLDS)

# Convert to the (train_positions, val_positions) pairs GridSearchCV expects.
# GridSearchCV indexes into X_train, so these must be positions WITHIN the training set.
position_in_train = {row: i for i, row in enumerate(train_idx_balanced)}
cv_folds_balanced = []
for validation_rows in validation_folds:
    val_positions = np.array([position_in_train[r] for r in validation_rows])
    train_positions = np.setdiff1d(np.arange(len(train_idx_balanced)), val_positions)
    cv_folds_balanced.append((train_positions, val_positions))

print(f"Built {len(cv_folds_balanced)} folds.")

## 10. Old vs new CV folds

In [ ]:
fold_summary_rows = []

# --- OLD: GroupKFold on the pure Murcko training set ---
old_group_series = df["scaffold"]
group_kfold = GroupKFold(n_splits=N_FOLDS)
for fold_number, (train_part, val_part) in enumerate(
    group_kfold.split(train_idx_old, groups=old_group_series.iloc[train_idx_old])
):
    val_rows = train_idx_old[val_part]
    train_rows = train_idx_old[train_part]
    fold_summary_rows.append({
        "Split method": "OLD  GroupKFold / pure Murcko",
        "Fold": fold_number,
        "n_train": len(train_rows),
        "n_val": len(val_rows),
        "train scaffolds": old_group_series.iloc[train_rows].nunique(),
        "val scaffolds": old_group_series.iloc[val_rows].nunique(),
        "overlap": len(set(old_group_series.iloc[train_rows]) & set(old_group_series.iloc[val_rows])),
        "val % of train": round(100 * len(val_rows) / len(train_idx_old), 1),
    })

# --- NEW: greedy balanced folds on the hybrid training set ---
for fold_number, val_rows in enumerate(validation_folds):
    train_rows = np.setdiff1d(train_idx_balanced, val_rows)
    fold_summary_rows.append({
        "Split method": "NEW  greedy balanced / hybrid",
        "Fold": fold_number,
        "n_train": len(train_rows),
        "n_val": len(val_rows),
        "train scaffolds": df["group"].iloc[train_rows].nunique(),
        "val scaffolds": df["group"].iloc[val_rows].nunique(),
        "overlap": len(set(df["group"].iloc[train_rows]) & set(df["group"].iloc[val_rows])),
        "val % of train": round(100 * len(val_rows) / len(train_idx_balanced), 1),
    })

fold_summary = pd.DataFrame(fold_summary_rows)
print(fold_summary.to_string(index=False))

Old folds spanned 13.7–36.8% of training, two of them a single scaffold each. New folds
span 19.5–22.0%.

Fold 0 remains a single-group fold: benzene, 1,753 molecules, 22.0% of training. Unlike
the acyclic block, benzene is a genuine shared Murcko scaffold, so splitting it would
break the disjointness guarantee. 22.0% against an ideal 20.0% is the floor for any
method keeping whole scaffolds together on this dataset.

### Target distribution across folds

The folds were built without ever looking at `y`. We verify here that this did not
accidentally produce a fold with a badly skewed target distribution.

In [ ]:
print("Train/test target distribution:")
for name, tr, te in [("OLD", train_idx_old, test_idx_old),
                     ("NEW", train_idx_balanced, test_idx_balanced)]:
    print(f"  {name}: train mean {y_all[tr].mean():+.3f} (std {y_all[tr].std():.3f}) | "
          f"test mean {y_all[te].mean():+.3f} (std {y_all[te].std():.3f}) | "
          f"shift {y_all[te].mean() - y_all[tr].mean():+.3f}")

print("\nNEW validation fold target distributions:")
for fold_number, val_rows in enumerate(validation_folds):
    print(f"  fold {fold_number}: n={len(val_rows):5d}  "
          f"mean {y_all[val_rows].mean():+.3f}  std {y_all[val_rows].std():.3f}")

## 11. Figures for the split analysis

These four depend only on the split, not on any model, so they render in seconds.
Each is written beside this notebook as a vector `.pdf`.

In [ ]:
# ---- Figure 2: CV fold balance, old vs new ----
old_folds = fold_summary[fold_summary["Split method"].str.startswith("OLD")]
new_folds = fold_summary[fold_summary["Split method"].str.startswith("NEW")]

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(N_FOLDS)
width = 0.38
ax.bar(x - width / 2, old_folds["val % of train"], width,
       label="OLD  pure Murcko + GroupKFold", color="#c44e52")
ax.bar(x + width / 2, new_folds["val % of train"], width,
       label="NEW  hybrid + greedy balanced", color="#4c72b0")
ax.axhline(100 / N_FOLDS, color="k", linestyle="--", linewidth=1,
           label=f"ideal ({100 / N_FOLDS:.0f}%)")

for xi, v in zip(x - width / 2, old_folds["val % of train"]):
    ax.text(xi, v + 0.6, f"{v:.1f}", ha="center", fontsize=9)
for xi, v in zip(x + width / 2, new_folds["val % of train"]):
    ax.text(xi, v + 0.6, f"{v:.1f}", ha="center", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([f"Fold {i}" for i in range(N_FOLDS)])
ax.set_ylabel("Validation fold size (% of training set)")
ax.set_title("Cross-validation fold balance")
ax.legend()
plt.tight_layout()
save_figure(fig, "02_fold_balance")
plt.show()

In [ ]:
# ---- Figure 3: train/test balance, old vs new ----
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
labels = ["OLD", "NEW"]
x = np.arange(2)

old_train_counts = df["scaffold"].iloc[train_idx_old].value_counts()
old_test_counts = df["scaffold"].iloc[test_idx_old].value_counts()
new_train_counts = df["group"].iloc[train_idx_balanced].value_counts()
new_test_counts = df["group"].iloc[test_idx_balanced].value_counts()

molecule_share = [100 * len(train_idx_old) / n_total,
                  100 * len(train_idx_balanced) / n_total]
group_share = [100 * len(old_train_counts) / (len(old_train_counts) + len(old_test_counts)),
               100 * len(new_train_counts) / (len(new_train_counts) + len(new_test_counts))]

axes[0].bar(x - 0.2, molecule_share, 0.4, label="molecules", color="#4c72b0")
axes[0].bar(x + 0.2, group_share, 0.4, label="groups", color="#dd8452")
axes[0].axhline(80, color="k", linestyle="--", linewidth=1, label="target 80%")
axes[0].set_xticks(x); axes[0].set_xticklabels(labels)
axes[0].set_ylabel("% assigned to train")
axes[0].set_title("(a) Molecule vs group balance")
axes[0].legend(fontsize=9)

axes[1].bar(x - 0.2, [old_train_counts.head(10).sum() / len(train_idx_old),
                      new_train_counts.head(10).sum() / len(train_idx_balanced)],
            0.4, label="train", color="#4c72b0")
axes[1].bar(x + 0.2, [old_test_counts.head(10).sum() / len(test_idx_old),
                      new_test_counts.head(10).sum() / len(test_idx_balanced)],
            0.4, label="test", color="#dd8452")
axes[1].set_xticks(x); axes[1].set_xticklabels(labels)
axes[1].set_ylabel("Fraction of molecules in top-10 groups")
axes[1].set_title("(b) Concentration in largest groups")
axes[1].legend(fontsize=9)

axes[2].bar(x - 0.2, [int((old_train_counts == 1).sum()), int((new_train_counts == 1).sum())],
            0.4, label="train", color="#4c72b0")
axes[2].bar(x + 0.2, [int((old_test_counts == 1).sum()), int((new_test_counts == 1).sum())],
            0.4, label="test", color="#dd8452")
axes[2].set_xticks(x); axes[2].set_xticklabels(labels)
axes[2].set_ylabel("Number of singleton groups")
axes[2].set_title("(c) Singleton groups per side")
axes[2].legend(fontsize=9)

plt.tight_layout()
save_figure(fig, "03_train_test_balance")
plt.show()

In [ ]:
# ---- Figure 4: group-size distribution before and after hybrid grouping ----
murcko_sizes = scaffold_counts.values
hybrid_group_sizes = np.array(sorted((len(v) for v in scaffold_to_indices.values()), reverse=True))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

bins = np.logspace(0, np.log10(max(murcko_sizes.max(), hybrid_group_sizes.max())), 40)
axes[0].hist(murcko_sizes, bins=bins, alpha=0.65, label="pure Murcko", color="#c44e52")
axes[0].hist(hybrid_group_sizes, bins=bins, alpha=0.65, label="hybrid", color="#4c72b0")
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("Molecules per group"); axes[0].set_ylabel("Number of groups")
axes[0].set_title("(a) Group size distribution")
axes[0].legend()

top_n = 15
axes[1].plot(range(1, top_n + 1), murcko_sizes[:top_n], "o-", label="pure Murcko", color="#c44e52")
axes[1].plot(range(1, top_n + 1), hybrid_group_sizes[:top_n], "s-", label="hybrid", color="#4c72b0")
axes[1].annotate("acyclic block\n(2,940) dissolved", xy=(1, murcko_sizes[0]),
                 xytext=(3.0, murcko_sizes[0] * 0.80), fontsize=9,
                 arrowprops=dict(arrowstyle="->", color="#c44e52"))
axes[1].annotate("benzene (1,753)\nkept intact", xy=(1, hybrid_group_sizes[0]),
                 xytext=(4.5, hybrid_group_sizes[0] * 0.35), fontsize=9,
                 arrowprops=dict(arrowstyle="->", color="#4c72b0"))
axes[1].set_yscale("log")
axes[1].set_xlabel("Group rank (largest first)"); axes[1].set_ylabel("Molecules in group")
axes[1].set_title(f"(b) {top_n} largest groups")
axes[1].legend()

plt.tight_layout()
save_figure(fig, "04_group_size_distribution")
plt.show()

In [ ]:
# ---- Figure 5: how the acyclic similarity threshold was chosen ----
cyclic_singleton_pct = 100 * (cyclic_sizes == 1).mean()
cyclic_p90 = np.percentile(cyclic_sizes, 90)
cyclic_mean_excl_benzene = cyclic_sizes[1:].mean()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
sims = threshold_sweep["similarity"]

for ax, column, reference, title in [
    (axes[0], "singleton %", cyclic_singleton_pct, "(a) Singleton fraction"),
    (axes[1], "p90", cyclic_p90, "(b) 90th percentile group size"),
    (axes[2], "mean", cyclic_mean_excl_benzene, "(c) Mean group size"),
]:
    ax.plot(sims, threshold_sweep[column], "o-", color="#4c72b0",
            label="acyclic Butina clusters")
    ax.axhline(reference, color="#c44e52", linestyle="--", linewidth=1.5,
               label="cyclic Murcko reference")
    ax.axvline(ACYCLIC_SIMILARITY_THRESHOLD, color="k", linestyle=":", linewidth=1.5,
               label=f"chosen = {ACYCLIC_SIMILARITY_THRESHOLD}")
    ax.invert_xaxis()
    ax.set_xlabel("Tanimoto similarity threshold")
    ax.set_title(title)
    ax.legend(fontsize=8)

axes[0].set_ylabel("% of clusters that are singletons")
axes[1].set_ylabel("molecules")
axes[2].set_ylabel("molecules")
fig.suptitle("Threshold chosen to match Murcko granularity — never from model error", y=1.03)
plt.tight_layout()
save_figure(fig, "05_threshold_selection")
plt.show()

## 12. Feature matrices

Two representations, computed exactly as in the previous notebooks so the only thing
that has changed is the split.

In [ ]:
descriptor_list = Descriptors._descList
print(f"RDKit descriptors available: {len(descriptor_list)}")

t0 = time.time()
descriptor_rows = []
for mol in mols:
    row = []
    for _, function in descriptor_list:
        try:
            row.append(function(mol))
        except Exception:
            row.append(np.nan)
    descriptor_rows.append(row)

X_rdkit_full = pd.DataFrame(
    descriptor_rows, columns=[name for name, _ in descriptor_list], index=df.index
)
print(f"Raw descriptor matrix: {X_rdkit_full.shape}  ({time.time() - t0:.1f}s)")

# Same cleaning as before: inf -> NaN, drop any column with NaN, drop constant columns.
X_rdkit_full = X_rdkit_full.replace([np.inf, -np.inf], np.nan)
X_rdkit_full = X_rdkit_full.dropna(axis=1)

# A few descriptors are finite but astronomically large. Ipc (information content) grows
# factorially with molecule size and reaches ~1e158 on this dataset. XGBoost stores
# features as float32, whose maximum is ~3.4e38, so such a column makes training fail
# with "Input data contains inf or a value too large". Drop anything unrepresentable.
float32_max = np.finfo(np.float32).max
oversized_columns = X_rdkit_full.columns[X_rdkit_full.abs().max() > float32_max]
if len(oversized_columns) > 0:
    print(f"Dropping {len(oversized_columns)} column(s) beyond float32 range: "
          f"{list(oversized_columns)}")
    X_rdkit_full = X_rdkit_full.drop(columns=oversized_columns)

X_rdkit_full = X_rdkit_full.loc[:, X_rdkit_full.std() > 0]
print(f"After cleaning       : {X_rdkit_full.shape}")

In [ ]:
t0 = time.time()
morgan_rows = np.zeros((len(mols), MORGAN_NBITS), dtype=np.int8)
for i, mol in enumerate(mols):
    fp = morgan_generator.GetFingerprint(mol)
    DataStructs.ConvertToNumpyArray(fp, morgan_rows[i])

X_morgan_full = pd.DataFrame(
    morgan_rows, columns=[f"bit_{i}" for i in range(MORGAN_NBITS)], index=df.index
)
print(f"Morgan matrix: {X_morgan_full.shape}  ({time.time() - t0:.1f}s)")
print(f"Bit density  : {morgan_rows.mean():.4f}")

### Representation 3 — RDKit descriptors + Morgan concatenated

Side-by-side concatenation along the feature axis. Both frames share `df.index`, so
`pd.concat(axis=1)` aligns them by molecule automatically.

In [ ]:
t0 = time.time()
X_combined_full = pd.concat([X_rdkit_full, X_morgan_full], axis=1)
print(f"Concatenation runtime: {time.time() - t0:.1f} s")

print(f"RDKit    : {X_rdkit_full.shape}")
print(f"Morgan   : {X_morgan_full.shape}")
print(f"Combined : {X_combined_full.shape}")
print(f"Any NaN in combined: {X_combined_full.isna().any().any()}")
print(f"First 3 columns : {list(X_combined_full.columns[:3])}")
print(f"Last  3 columns : {list(X_combined_full.columns[-3:])}")

In [ ]:
X_train_rdkit = X_rdkit_full.iloc[train_idx_balanced]
X_test_rdkit = X_rdkit_full.iloc[test_idx_balanced]
X_train_morgan = X_morgan_full.iloc[train_idx_balanced]
X_test_morgan = X_morgan_full.iloc[test_idx_balanced]
X_train_combined = X_combined_full.iloc[train_idx_balanced]
X_test_combined = X_combined_full.iloc[test_idx_balanced]

y_train = y_all[train_idx_balanced]
y_test = y_all[test_idx_balanced]

print(f"RDKit  train {X_train_rdkit.shape}  test {X_test_rdkit.shape}")
print(f"Morgan train {X_train_morgan.shape}  test {X_test_morgan.shape}")
print(f"Comb.  train {X_train_combined.shape}  test {X_test_combined.shape}")
print(f"y      train {y_train.shape}  test {y_test.shape}")

## 13. Shared XGBoost setup

Identical grid, seed and parallelism settings to `AqSolDB_representations.ipynb`. The
only difference between this notebook and that one is the split and the CV folds.

In [ ]:
xgb_param_grid = {
    "n_estimators":     [100, 200, 500],
    "learning_rate":    [0.01, 0.05, 0.1],
    "max_depth":        [2, 3, 4],
    "subsample":        [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}
n_combinations = np.prod([len(v) for v in xgb_param_grid.values()])
print(f"Grid size: {n_combinations} combinations")
print(f"Total fits per experiment: {n_combinations * N_FOLDS}")


def run_experiment(name, X_train, X_test, y_train, y_test, cv_folds):
    """Fit XGBoost with GridSearchCV over the given folds and report the standard metrics."""
    t0 = time.time()
    model = XGBRegressor(
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=1,
        tree_method="hist",
    )
    search = GridSearchCV(
        estimator=model,
        param_grid=xgb_param_grid,
        cv=cv_folds,
        scoring="neg_root_mean_squared_error",
        n_jobs=2,
        return_train_score=True,
    )
    search.fit(X_train, y_train)
    runtime = time.time() - t0

    y_pred = search.best_estimator_.predict(X_test)
    result = {
        "Representation": name,
        "N features": X_train.shape[1],
        "CV RMSE": -search.best_score_,
        "Test RMSE": root_mean_squared_error(y_test, y_pred),
        "Test MAE": mean_absolute_error(y_test, y_pred),
        "Test R2": r2_score(y_test, y_pred),
        "NRMSE": root_mean_squared_error(y_test, y_pred) / y_test.std(),
        "Runtime (s)": runtime,
    }

    print(f"=== {name} ===")
    print(f"Best hyperparameters: {search.best_params_}")
    for key in ["CV RMSE", "Test RMSE", "Test MAE", "Test R2", "NRMSE"]:
        print(f"  {key:<12}: {result[key]:.4f}")
    print(f"  {'Runtime':<12}: {runtime:.1f} s   ({runtime / 60:.1f} min)")
    return result, search, y_pred

## 14. RDKit descriptors + XGBoost

Run this one first and confirm it behaves sensibly before spending time on Morgan.

In [ ]:
cell_t0 = time.time()

result_rdkit, search_rdkit, y_pred_rdkit = run_experiment(
    "RDKit descriptors", X_train_rdkit, X_test_rdkit, y_train, y_test, cv_folds_balanced
)

print(f"\nTotal cell runtime: {time.time() - cell_t0:.1f} s "
      f"({(time.time() - cell_t0) / 60:.1f} min)")


## 15. Morgan fingerprints + XGBoost

Same split, same folds, same grid — only the feature matrix changes.

In [ ]:
cell_t0 = time.time()

result_morgan, search_morgan, y_pred_morgan = run_experiment(
    "Morgan fingerprints", X_train_morgan, X_test_morgan, y_train, y_test, cv_folds_balanced
)

print(f"\nTotal cell runtime: {time.time() - cell_t0:.1f} s "
      f"({(time.time() - cell_t0) / 60:.1f} min)")


## 16. RDKit + Morgan combined

This experiment was never run on AqSolDB before — it was skipped in
`AqSolDB_representations.ipynb` (`RUN_COMBINED=False`), so there is no old AqSolDB
baseline to compare against. On ESOL the combined representation scored 0.904 test
RMSE versus 0.886 for RDKit alone, i.e. concatenation was marginally *worse* than
descriptors on their own. This is the longest run of the three.

In [ ]:
cell_t0 = time.time()

result_combined, search_combined, y_pred_combined = run_experiment(
    "RDKit + Morgan", X_train_combined, X_test_combined, y_train, y_test, cv_folds_balanced
)

print(f"\nTotal cell runtime: {time.time() - cell_t0:.1f} s "
      f"({(time.time() - cell_t0) / 60:.1f} min)")


## 17. Results

In [ ]:
results_new = pd.DataFrame([result_rdkit, result_morgan, result_combined])
print("NEW — hybrid structure-aware split, group-balanced CV folds:")
print(results_new.round(4).to_string(index=False))

results_old = pd.DataFrame([
    {"Representation": "RDKit descriptors", "N features": 202,
     "CV RMSE": 1.1100, "Test RMSE": 1.2665, "Test MAE": 0.9450, "Test R2": 0.6829},
    {"Representation": "Morgan fingerprints", "N features": 2048,
     "CV RMSE": 1.8059, "Test RMSE": 1.7935, "Test MAE": 1.3798, "Test R2": 0.3642},
])
print("\nOLD — pure Murcko largest-first split, GroupKFold (baseline, unchanged):")
print(results_old.round(4).to_string(index=False))

In [ ]:
rmse_new = results_new.set_index("Representation")["Test RMSE"]
rmse_old = results_old.set_index("Representation")["Test RMSE"]

gap_old = rmse_old["Morgan fingerprints"] - rmse_old["RDKit descriptors"]
gap_new = rmse_new["Morgan fingerprints"] - rmse_new["RDKit descriptors"]

print("Morgan-minus-RDKit test RMSE gap")
print(f"  OLD split : {rmse_old['Morgan fingerprints']:.3f} - {rmse_old['RDKit descriptors']:.3f} = {gap_old:+.3f}")
print(f"  NEW split : {rmse_new['Morgan fingerprints']:.3f} - {rmse_new['RDKit descriptors']:.3f} = {gap_new:+.3f}")
print(f"  Change    : {gap_new - gap_old:+.3f}")

# Does concatenating Morgan onto RDKit help, hurt, or do nothing?
delta_combined = rmse_new["RDKit + Morgan"] - rmse_new["RDKit descriptors"]
print("\nDoes adding Morgan bits to RDKit descriptors help?")
print(f"  RDKit alone      : {rmse_new['RDKit descriptors']:.3f}")
print(f"  RDKit + Morgan   : {rmse_new['RDKit + Morgan']:.3f}")
print(f"  Difference       : {delta_combined:+.3f}  "
      f"({'combined is worse' if delta_combined > 0 else 'combined is better'})")
print(f"  ESOL reference   : +0.018 (combined was marginally worse there too)")

In [ ]:
# ---- Figure 6: CV honesty — does CV predict held-out error? ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, frame, title in [
    (axes[0], results_old, "(a) OLD  pure Murcko + GroupKFold"),
    (axes[1], results_new, "(b) NEW  hybrid + greedy balanced"),
]:
    names = frame["Representation"]
    cv_vals = frame["CV RMSE"].values
    test_vals = frame["Test RMSE"].values
    x = np.arange(len(names))
    ax.bar(x - 0.2, cv_vals, 0.4, label="CV RMSE", color="#4c72b0")
    ax.bar(x + 0.2, test_vals, 0.4, label="Test RMSE", color="#dd8452")
    for xi, c, t in zip(x, cv_vals, test_vals):
        ax.text(xi, max(c, t) + 0.05, f"gap {t - c:+.3f}", ha="center", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([n.replace(" ", "\n") for n in names], fontsize=9)
    ax.set_ylabel("RMSE (log units)")
    ax.set_ylim(0, max(list(cv_vals) + list(test_vals)) * 1.3)
    ax.set_title(title)
    ax.legend(fontsize=9)

fig.suptitle("A small CV-test gap means cross-validation is predicting held-out error", y=1.02)
plt.tight_layout()
save_figure(fig, "06_cv_test_gap")
plt.show()

### Diagnostic plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, y_pred, title in [
    (axes[0], y_pred_rdkit, f"RDKit  (RMSE={result_rdkit['Test RMSE']:.3f})"),
    (axes[1], y_pred_morgan, f"Morgan (RMSE={result_morgan['Test RMSE']:.3f})"),
    (axes[2], y_pred_combined, f"RDKit+Morgan (RMSE={result_combined['Test RMSE']:.3f})"),
]:
    ax.scatter(y_test, y_pred, alpha=0.3, s=12)
    lo, hi = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
    ax.plot([lo, hi], [lo, hi], "r--", linewidth=1, label="y = x")
    ax.set_xlabel("Measured logS")
    ax.set_ylabel("Predicted logS")
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
save_figure(fig, "07_predicted_vs_actual")
plt.show()

## 18. Conclusions

The old split was scaffold-disjoint but strongly imbalanced, because of the dataset's
scaffold-size distribution combined with largest-first assignment. Replacing the
indivisible acyclic block with similarity clusters, and largest-first with a
balance-scored assignment, gives:

- group counts tracking molecule counts (~80/20) instead of 15/85
- singleton groups on both sides instead of only in test
- comparable top-10 concentration in train and test instead of 0.70 vs 0.02
- CV folds spanning 19.5–22.0% of training instead of 13.7–36.8%

Disjointness is preserved exactly: `train_scaffolds ∩ test_scaffolds` is empty and no
group spans CV folds.

**Comparing the numbers.** The new test RMSE is not comparable to the old 1.267 as a
measure of model quality — the two test sets contain different molecules with different
group structure, so they answer different questions. A change in either direction is
expected. The split was chosen on structural criteria before any model was fitted, so
test performance played no part in selecting it.

**Combined representation.** RDKit + Morgan is new for AqSolDB; it was skipped in
`AqSolDB_representations.ipynb`, so there is no old baseline. Read it against
RDKit-alone within this notebook. Note the combined model carries 2,250 features against
202, so a small difference either way is weak evidence that the extra bits add
independent signal.

For the write-up, this is a *hybrid structure-aware split: Bemis–Murcko scaffolds for
cyclic molecules and fingerprint-similarity clusters for acyclic molecules*.

## 19. Why the two representations behave differently

The gap between descriptors and fingerprints follows from what each encodes.

**RDKit descriptors** are aggregate, whole-molecule, real-valued quantities — `MolLogP`,
`TPSA`, `MolWt`, `BertzCT`, the Chi and EState indices, the `fr_*` counts. Each pools
over the whole molecule and has a meaningful ordering: logP 2 < logP 3 < logP 4 is a
real statement about hydrophobicity.

**Morgan (ECFP4)** is a local, hashed, binary set-membership vector. Bit *k* means "this
molecule contains a circular atom environment, radius ≤ 2, whose identifier hashes to
bucket *k*". A bit has no intrinsic meaning, unrelated substructures can collide into
one, and there is no ordering — a tree can only ask present/absent.

Three consequences under a structure-aware split:

- **Transferability.** Crippen logP comes from an atom-contribution scheme, so an unseen
  scaffold still gets a sensible value. A Morgan bit is informative only if that exact
  substructure appeared in training often enough to learn a split on. On novel chemistry
  the useful bits are simply off.
- **Sparsity.** At ~1% bit density, roughly 18 of 2,048 bits are set per molecule, so
  most bits are rare and have few positive examples to learn from.
- **Task match.** Solubility is a bulk thermodynamic property — hydrophobicity,
  polarity, size, crystal packing. Descriptors encode those directly; Morgan encodes
  local topology from which they must be reconstructed.

This is consistent with the SHAP result in `Scaffold_split.ipynb`, where logP's mean
|SHAP| rose from 1.335 under a random split to 1.611 under a scaffold split — the model
leans harder on transferable physics when the test chemistry is unfamiliar.

Morgan is not a worse representation in general, it is mismatched to this task. Where a
specific substructure is the causal signal — a toxicophore, a reactive group, a binding
motif — fingerprints routinely beat descriptors. It also appears data-hungry rather than
uninformative: test R² was 0.333 on ESOL's 902 training molecules and improves with more
training data and a less exotic test set.

Concatenation need not help either. On ESOL, RDKit + Morgan gave 0.904 against 0.886 for
RDKit alone; adding ~2,048 sparse, mostly uninformative columns to ~200 informative ones
dilutes the pool that split-finding searches. That is dilution, not proof the bits are
empty.

> Descriptors encode properties that mean the same thing on unfamiliar chemistry;
> fingerprint bits encode specific substructures that are not present there.